# Creating Constraints

A constraint relates two expressions with `≤`, `≥`, or `=`. In linopy you write the constraint in natural mathematical form, hand it to `model.add_constraints`, and it becomes part of the model. The same conventions you've already seen for expressions — broadcasting by dimension name, explicit alignment when coordinates can mismatch — carry over directly.

This page covers four topics:

- writing and assigning constraints with the `<=` / `>=` / `==` operators;
- the explicit-alignment forms `.le()` / `.ge()` / `.eq()` for when coordinate ranges might disagree;
- the CSR backend — a memory-efficient alternative storage format for large models.

In [ ]:
from linopy import Model

m = Model()
x = m.add_variables(name="x")

## Writing constraints

Consider the constraint $x \le \tfrac{10}{3}$. linopy accepts every algebraically equivalent rearrangement — write whichever form is most natural for the problem:

$$
x \le \tfrac{10}{3} \qquad\Longleftrightarrow\qquad 3x \le 10 \qquad\Longleftrightarrow\qquad 3x - 10 \le 0
$$

Applying `<=`, `>=`, or `==` to an expression builds an **unassigned** constraint — one that exists as a value but isn't yet part of any model:

In [ ]:
con = 3 * x <= 10
con

An unassigned constraint exposes its left-hand side, right-hand side, and sign so you can inspect what the math reduced to before committing it to the model:

In [ ]:
con.lhs

In [ ]:
con.rhs

Pass the unassigned constraint to `m.add_constraints` to attach it to the model. Always set `name=` — modifying or looking up the constraint later relies on it:

In [ ]:
c = m.add_constraints(con, name="capacity")
c

You can also write the constraint inline, without the named intermediate — same end result:

In [ ]:
m.add_constraints(3 * x <= 10, name="capacity_inline")

The returned `Constraint` carries the reference labels into the optimization model and forwards attribute access to `.lhs`, `.rhs`, and `.sign`:

In [ ]:
c.lhs

### Automatic LHS/RHS separation

linopy doesn't mind which side of the inequality the constants are on. Constants written on the variable side get pulled across automatically, so every committed constraint ends up with variables on the left and the constant on the right:

In [ ]:
3 * x - 10

In [ ]:
3 * x - 10 <= 0

Both forms produce the same final constraint, with consistent LHS/RHS shape.

### Accessing constraints on the model

All constraints added to the model live in `m.constraints` and are looked up by name:

In [ ]:
m.constraints

In [ ]:
m.constraints["capacity"]

## Explicit alignment with `.le()`, `.ge()`, `.eq()`

The `<=`, `>=`, `==` operators are convenient but they inherit the coordinate-alignment behaviour shown in :doc:`creating-expressions`: when the two sides share a dim name with mismatched coordinates, the left operand's coordinates win silently.

For constraints where that matters — typically when LHS and RHS come from different data sources — use the method forms `.le()`, `.ge()`, and `.eq()` instead. They take an explicit `join` (`"inner"`, `"outer"`, `"left"`, `"right"`) so you control how the two sides line up:

In [ ]:
import pandas as pd

a = m.add_variables(coords=[pd.Index(range(0, 10), name="time")], name="a")
b = m.add_variables(coords=[pd.Index(range(5, 15), name="time")], name="b")

# Inner join: constraint applies only on the time steps both variables cover.
a.le(b, join="inner")

In [ ]:
# Outer join: union of the two ranges, with partial terms where one side is missing.
a.le(b, join="outer")

The :doc:`coordinate-alignment` guide walks through every join option in depth.

## The CSR backend

By default, linopy stores each constraint as an `xarray.Dataset` (`Constraint`). This is flexible and supports full label-based indexing, but can be memory-heavy when constraints carry many terms.

For large models, the **CSR backend** (`CSRConstraint`) stores coefficients as a [scipy CSR sparse matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_array.html), with flat numpy arrays for the right-hand side and signs. It reduces memory usage by up to **90%** and speeds up matrix generation for direct solver APIs by **30–120×**.

`CSRConstraint` is **immutable** — once frozen, the constraint data can't be modified in place. Conversion back to the mutable xarray-backed `Constraint` is always available.

### Freezing individual constraints

Pass `freeze=True` to `add_constraints` to store a single constraint in CSR format:

In [ ]:
import numpy as np

m2 = Model()
y = m2.add_variables(coords=[np.arange(100)], name="y")

m2.add_constraints(y <= 10, name="upper", freeze=True)

print(type(m2.constraints["upper"]))
m2.constraints["upper"]

### Freezing all constraints globally

Set `freeze_constraints=True` on the `Model` to automatically freeze every constraint added via `add_constraints`:

In [ ]:
m3 = Model(freeze_constraints=True)
z3 = m3.add_variables(coords=[np.arange(50)], name="z")
m3.add_constraints(z3 >= 0, name="lower")
m3.add_constraints(z3 <= 100, name="upper")

print(type(m3.constraints["lower"]))
print(type(m3.constraints["upper"]))

### Converting between representations

Use `.freeze()` and `.mutable()` to convert between the two representations. The conversion is lossless:

In [ ]:
frozen = m3.constraints["lower"]
print(f"Frozen type:    {type(frozen).__name__}")

thawed = frozen.mutable()
print(f"Mutable type:   {type(thawed).__name__}")

refrozen = thawed.freeze()
print(f"Re-frozen type: {type(refrozen).__name__}")

### API differences from `Constraint`

`CSRConstraint` deliberately exposes a narrower API than the xarray-backed `Constraint`:

- **No in-place mutation.** Setters such as `con.coeffs = ...`, `con.vars = ...`, `con.sign = ...`, `con.rhs = ...`, and `con.lhs = ...` are only available on `Constraint`.
- **No label-based indexing.** `con.loc[...]` is only available on `Constraint`.
- **Accessing `.coeffs` / `.vars` triggers reconstruction.** On a `CSRConstraint` these properties rebuild the full xarray `Dataset` on demand and emit a `PerformanceWarning`. For solver-oriented workflows prefer `con.to_matrix()` or work with the CSR data directly.

If you need any of the above, call `.mutable()` first to get a `Constraint`:

```python
con = m.constraints["my_constraint"].mutable()
con.loc[{"time": 0}]  # label-based indexing now available
con.rhs = 5           # mutation now available
```

### When to use the CSR backend

The CSR backend is most beneficial when:

- the model has **many constraints with many terms**;
- **memory** is a bottleneck;
- you're using a **direct solver API** (e.g. HiGHS, Gurobi Python bindings) rather than file-based I/O.

For small models the overhead is negligible and the default xarray-backed `Constraint` is perfectly fine.

If you don't need variable and constraint names exported to the solver (e.g. for batch solves), you can disable name export for extra speed:

```python
m = Model(freeze_constraints=True, set_names_in_solver_io=False)
```

## Where to next

- :doc:`coordinate-alignment` — the deep dive on `join` semantics for both expressions and constraints.
- :doc:`manipulating-models` — modifying constraints (sign, RHS, removing) after they're on the model.